In [ ]:


import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Try to import nibabel for coordinate transformations
try:
    import nibabel as nib
    from nibabel import affines
    NIBABEL_AVAILABLE = True
except ImportError:
    NIBABEL_AVAILABLE = False
    print("Warning: nibabel not available. Coordinate space conversion limited.")

In [ ]:
class HabenulaPeakAnalysis:
    def __init__(self, your_peaks_file, literature_peaks_file):
        """
        Initialize the analysis with peak files

        Parameters:
        -----------
        your_peaks_file : str
            Path to TSV file with your peaks (columns: x, y, z, and optionally space, voxel_size)
        literature_peaks_file : str
            Path to TSV file with literature peaks (columns: x, y, z, and optionally space, voxel_size)
        """
        self.your_peaks = pd.read_csv(your_peaks_file, sep="\t")
        self.lit_peaks = pd.read_csv(literature_peaks_file, sep="\t")

        # Validate required columns
        required_cols = ["x", "y", "z"]
        for df, name in [
            (self.your_peaks, "your peaks"),
            (self.lit_peaks, "literature peaks"),
        ]:
            missing = [col for col in required_cols if col not in df.columns]
            if missing:
                raise ValueError(f"Missing columns in {name}: {missing}")

        print(f"Loaded {len(self.your_peaks)} of your peaks")
        print(f"Loaded {len(self.lit_peaks)} literature peaks")

        # Check for coordinate space and voxel size columns
        self._check_coordinate_info()

    def _check_coordinate_info(self):
        """Check and standardize coordinate space information"""

        # Add default coordinate space if not specified
        if "space" not in self.your_peaks.columns:
            self.your_peaks["space"] = "MNI"
            print(
                "Warning: No coordinate space specified for your peaks. Assuming MNI."
            )

        if "space" not in self.lit_peaks.columns:
            self.lit_peaks["space"] = "MNI"
            print(
                "Warning: No coordinate space specified for literature peaks. Assuming MNI."
            )

        # Add default voxel size if not specified
        if "voxel_size" not in self.your_peaks.columns:
            self.your_peaks["voxel_size"] = 2.0  # Common fMRI voxel size
            print("Warning: No voxel size specified for your peaks. Assuming 2mm.")

        if "voxel_size" not in self.lit_peaks.columns:
            self.lit_peaks["voxel_size"] = 2.0
            print(
                "Warning: No voxel size specified for literature peaks. Assuming 2mm."
            )

    def convert_coordinates(self, from_space="TAL", to_space="MNI"):
        """
        Convert coordinates between Talairach and MNI space
        Uses the icbm_spm2tal transform (commonly used approximation)

        Parameters:
        -----------
        from_space : str
            Source coordinate space ('TAL' or 'MNI')
        to_space : str
            Target coordinate space ('TAL' or 'MNI')
        """

        if from_space == to_space:
            return

        # Simple linear transformation (Brett et al. 2002)
        # This is an approximation - for more precise conversion use nibabel transforms
        def tal_to_mni(coords):
            """Convert Talairach to MNI coordinates"""
            x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]

            # Different transforms for different brain regions
            mni_coords = np.zeros_like(coords)

            # Superior (z >= 0)
            superior = z >= 0
            if np.any(superior):
                mni_coords[superior, 0] = 0.99 * x[superior]
                mni_coords[superior, 1] = 0.9688 * y[superior] + 0.0460 * z[superior]
                mni_coords[superior, 2] = -0.0485 * y[superior] + 0.9189 * z[superior]

            # Inferior (z < 0)
            inferior = z < 0
            if np.any(inferior):
                mni_coords[inferior, 0] = 0.99 * x[inferior]
                mni_coords[inferior, 1] = 0.9688 * y[inferior] + 0.0460 * z[inferior]
                mni_coords[inferior, 2] = -0.0485 * y[inferior] + 0.8390 * z[inferior]

            return mni_coords

        def mni_to_tal(coords):
            """Convert MNI to Talairach coordinates (inverse transform)"""
            x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]

            tal_coords = np.zeros_like(coords)
            tal_coords[:, 0] = x / 0.99

            # Approximate inverse (simplified)
            tal_coords[:, 1] = (y - 0.0460 * z) / 0.9688
            tal_coords[:, 2] = z / 0.9189  # Simplified for superior regions

            return tal_coords

        # Apply conversions where needed
        for df, name in [
            (self.your_peaks, "your peaks"),
            (self.lit_peaks, "literature peaks"),
        ]:
            mask = df["space"] == from_space
            if np.any(mask):
                coords = df.loc[mask, ["x", "y", "z"]].values

                if from_space == "TAL" and to_space == "MNI":
                    new_coords = tal_to_mni(coords)
                elif from_space == "MNI" and to_space == "TAL":
                    new_coords = mni_to_tal(coords)
                else:
                    raise ValueError(
                        f"Conversion from {from_space} to {to_space} not supported"
                    )

                df.loc[mask, ["x", "y", "z"]] = new_coords
                df.loc[mask, "space"] = to_space

                print(
                    f"Converted {np.sum(mask)} {name} from {from_space} to {to_space}"
                )

    def standardize_coordinates(self, target_space="MNI"):
        """
        Standardize all coordinates to the same space

        Parameters:
        -----------
        target_space : str
            Target coordinate space ('MNI' or 'TAL')
        """
        print(f"\nStandardizing coordinates to {target_space} space...")

        # Check current coordinate spaces
        your_spaces = self.your_peaks["space"].unique()
        lit_spaces = self.lit_peaks["space"].unique()

        print(f"Your peaks coordinate spaces: {your_spaces}")
        print(f"Literature peaks coordinate spaces: {lit_spaces}")

        # Convert coordinates that aren't in target space
        for space in your_spaces:
            if space != target_space:
                self.convert_coordinates(from_space=space, to_space=target_space)

        for space in lit_spaces:
            if space != target_space:
                self.convert_coordinates(from_space=space, to_space=target_space)

        print(f"All coordinates now in {target_space} space")

    def create_spherical_roi(self, center, radius=8):
        """
        Create a spherical ROI around a center coordinate
        Returns a set of voxel coordinates within the sphere

        Parameters:
        -----------
        center : array-like
            Center coordinates [x, y, z]
        radius : float
            Sphere radius in mm
        """
        # Create a grid of coordinates around the center
        # Use 1mm resolution for accuracy
        x_range = np.arange(center[0] - radius, center[0] + radius + 1, 1)
        y_range = np.arange(center[1] - radius, center[1] + radius + 1, 1)
        z_range = np.arange(center[2] - radius, center[2] + radius + 1, 1)

        # Create meshgrid
        X, Y, Z = np.meshgrid(x_range, y_range, z_range, indexing="ij")
        coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

        # Calculate distances from center
        distances = np.sqrt(np.sum((coords - center) ** 2, axis=1))

        # Return coordinates within radius
        within_sphere = coords[distances <= radius]
        return set(map(tuple, within_sphere))

    def calculate_dice_coefficient(self, roi1, roi2):
        """
        Calculate Dice similarity coefficient between two ROI sets

        Parameters:
        -----------
        roi1, roi2 : set
            Sets of voxel coordinates

        Returns:
        --------
        float : Dice coefficient (0-1)
        """
        intersection = len(roi1.intersection(roi2))
        union_size = len(roi1) + len(roi2)

        if union_size == 0:
            return 0.0

        return 2.0 * intersection / union_size

    def compute_all_dice_coefficients(self, radius=8):
        """
        Compute Dice coefficients between all pairs of peaks

        Parameters:
        -----------
        radius : float
            Sphere radius in mm for ROI creation

        Returns:
        --------
        dict : Results containing Dice coefficients and statistics
        """
        print(f"\nComputing Dice coefficients with {radius}mm radius spheres...")

        # Ensure coordinates are standardized
        self.standardize_coordinates()

        # Get coordinate arrays
        your_coords = self.your_peaks[["x", "y", "z"]].values
        lit_coords = self.lit_peaks[["x", "y", "z"]].values

        # Create ROIs for all peaks
        print("Creating spherical ROIs...")
        your_rois = [self.create_spherical_roi(coord, radius) for coord in your_coords]
        lit_rois = [self.create_spherical_roi(coord, radius) for coord in lit_coords]

        # Compute Dice coefficients
        print("Computing Dice coefficients...")
        dice_matrix = np.zeros((len(your_coords), len(lit_coords)))

        for i, your_roi in enumerate(your_rois):
            for j, lit_roi in enumerate(lit_rois):
                dice_matrix[i, j] = self.calculate_dice_coefficient(your_roi, lit_roi)

        # Find best matches
        max_dice_per_your_peak = np.max(dice_matrix, axis=1)
        best_lit_match_idx = np.argmax(dice_matrix, axis=1)

        max_dice_per_lit_peak = np.max(dice_matrix, axis=0)
        best_your_match_idx = np.argmax(dice_matrix, axis=0)

        # Calculate distances to best matches
        distances_to_best = []
        for i, best_j in enumerate(best_lit_match_idx):
            dist = np.sqrt(np.sum((your_coords[i] - lit_coords[best_j]) ** 2))
            distances_to_best.append(dist)

        results = {
            "dice_matrix": dice_matrix,
            "max_dice_per_your_peak": max_dice_per_your_peak,
            "best_lit_match_idx": best_lit_match_idx,
            "max_dice_per_lit_peak": max_dice_per_lit_peak,
            "best_your_match_idx": best_your_match_idx,
            "distances_to_best": np.array(distances_to_best),
            "radius": radius,
            "mean_dice": np.mean(max_dice_per_your_peak),
            "median_dice": np.median(max_dice_per_your_peak),
            "min_dice": np.min(max_dice_per_your_peak),
            "max_dice": np.max(max_dice_per_your_peak),
            "std_dice": np.std(max_dice_per_your_peak),
        }

        return results

    def plot_results(self, results, save_path=None):
        """
        Create visualization plots of the Dice analysis results

        Parameters:
        -----------
        results : dict
            Results from compute_all_dice_coefficients()
        save_path : str, optional
            Path to save the plots
        """
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(
            "Habenula Peak Similarity Analysis", fontsize=16, fontweight="bold"
        )

        # Plot 1: Dice coefficient heatmap
        ax1 = axes[0, 0]
        im = ax1.imshow(results["dice_matrix"], cmap="viridis", aspect="auto")
        ax1.set_title("Dice Coefficient Matrix")
        ax1.set_xlabel("Literature Peaks")
        ax1.set_ylabel("Your Peaks")
        plt.colorbar(im, ax=ax1, label="Dice Coefficient")

        # Plot 2: Distribution of best Dice coefficients
        ax2 = axes[0, 1]
        ax2.hist(
            results["max_dice_per_your_peak"],
            bins=20,
            alpha=0.7,
            color="skyblue",
            edgecolor="black",
        )
        ax2.axvline(
            results["mean_dice"],
            color="red",
            linestyle="--",
            label=f"Mean: {results['mean_dice']:.3f}",
        )
        ax2.axvline(
            results["median_dice"],
            color="orange",
            linestyle="--",
            label=f"Median: {results['median_dice']:.3f}",
        )
        ax2.set_title("Distribution of Best Dice Coefficients")
        ax2.set_xlabel("Dice Coefficient")
        ax2.set_ylabel("Frequency")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Plot 3: Distance vs Dice coefficient scatter
        ax3 = axes[1, 0]
        scatter = ax3.scatter(
            results["distances_to_best"],
            results["max_dice_per_your_peak"],
            alpha=0.6,
            c=results["max_dice_per_your_peak"],
            cmap="viridis",
        )
        ax3.set_title("Distance vs Dice Coefficient")
        ax3.set_xlabel("Distance to Best Match (mm)")
        ax3.set_ylabel("Dice Coefficient")
        plt.colorbar(scatter, ax=ax3, label="Dice Coefficient")
        ax3.grid(True, alpha=0.3)

        # Plot 4: Summary statistics
        ax4 = axes[1, 1]
        stats_labels = ["Mean", "Median", "Min", "Max", "Std"]
        stats_values = [
            results["mean_dice"],
            results["median_dice"],
            results["min_dice"],
            results["max_dice"],
            results["std_dice"],
        ]

        bars = ax4.bar(
            stats_labels,
            stats_values,
            color=["skyblue", "lightgreen", "lightcoral", "gold", "plum"],
        )
        ax4.set_title("Dice Coefficient Summary Statistics")
        ax4.set_ylabel("Dice Coefficient")
        ax4.grid(True, alpha=0.3, axis="y")

        # Add value labels on bars
        for bar, value in zip(bars, stats_values):
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{value:.3f}",
                ha="center",
                va="bottom",
            )

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"Plots saved to: {save_path}")

        plt.show()

    def print_summary(self, results):
        """Print a summary of the analysis results"""
        print("\n" + "=" * 60)
        print("HABENULA PEAK SIMILARITY ANALYSIS SUMMARY")
        print("=" * 60)
        print(f"Number of your peaks: {len(self.your_peaks)}")
        print(f"Number of literature peaks: {len(self.lit_peaks)}")
        print(f"Sphere radius used: {results['radius']}mm")
        print()
        print("DICE COEFFICIENT STATISTICS:")
        print(f"  Mean Dice coefficient: {results['mean_dice']:.3f}")
        print(f"  Median Dice coefficient: {results['median_dice']:.3f}")
        print(f"  Standard deviation: {results['std_dice']:.3f}")
        print(f"  Range: {results['min_dice']:.3f} - {results['max_dice']:.3f}")
        print()
        print("DISTANCE STATISTICS:")
        print(
            f"  Mean distance to best match: {np.mean(results['distances_to_best']):.1f}mm"
        )
        print(
            f"  Median distance to best match: {np.median(results['distances_to_best']):.1f}mm"
        )
        print()

        # Interpretation guidelines
        print("INTERPRETATION GUIDELINES:")
        print("  Dice > 0.5: Good overlap")
        print("  Dice > 0.3: Moderate overlap")
        print("  Dice > 0.1: Some overlap")
        print("  Dice ≤ 0.1: Poor overlap")
        print()

        # Count peaks in each category
        good_overlap = np.sum(results["max_dice_per_your_peak"] > 0.5)
        moderate_overlap = np.sum(
            (results["max_dice_per_your_peak"] > 0.3)
            & (results["max_dice_per_your_peak"] <= 0.5)
        )
        some_overlap = np.sum(
            (results["max_dice_per_your_peak"] > 0.1)
            & (results["max_dice_per_your_peak"] <= 0.3)
        )
        poor_overlap = np.sum(results["max_dice_per_your_peak"] <= 0.1)

        print("OVERLAP CATEGORIES:")
        print(
            f"  Good overlap (>0.5): {good_overlap}/{len(self.your_peaks)} peaks ({good_overlap/len(self.your_peaks)*100:.1f}%)"
        )
        print(
            f"  Moderate overlap (0.3-0.5): {moderate_overlap}/{len(self.your_peaks)} peaks ({moderate_overlap/len(self.your_peaks)*100:.1f}%)"
        )
        print(
            f"  Some overlap (0.1-0.3): {some_overlap}/{len(self.your_peaks)} peaks ({some_overlap/len(self.your_peaks)*100:.1f}%)"
        )
        print(
            f"  Poor overlap (≤0.1): {poor_overlap}/{len(self.your_peaks)} peaks ({poor_overlap/len(self.your_peaks)*100:.1f}%)"
        )

        print("\n" + "=" * 60)

## Example Usage

Now you can use the `HabenulaPeakAnalysis` class to analyze your data. Modify the file paths below to point to your actual TSV files.

In [ ]:
# Enhanced version with proper voxel size handling
class VoxelAwareHabenulaPeakAnalysis(HabenulaPeakAnalysis):
    """Enhanced version that properly handles different voxel sizes"""
    
    def create_voxel_aware_spherical_roi(self, center, radius=8, voxel_size=1.0):
        """
        Create a spherical ROI using the actual voxel size of the data
        
        Parameters:
        -----------
        center : array-like
            Center coordinates [x, y, z]
        radius : float
            Sphere radius in mm
        voxel_size : float
            Voxel size in mm (0.7, 1.0, 2.0, etc.)
        """
        # Create a grid using the actual voxel size
        extent = radius + voxel_size  # Add one voxel buffer
        
        x_range = np.arange(center[0] - extent, center[0] + extent + voxel_size, voxel_size)
        y_range = np.arange(center[1] - extent, center[1] + extent + voxel_size, voxel_size)
        z_range = np.arange(center[2] - extent, center[2] + extent + voxel_size, voxel_size)
        
        # Round to avoid floating point precision issues
        x_range = np.round(x_range, 2)
        y_range = np.round(y_range, 2)
        z_range = np.round(z_range, 2)
        
        # Create meshgrid
        X, Y, Z = np.meshgrid(x_range, y_range, z_range, indexing="ij")
        coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
        
        # Calculate distances from center
        distances = np.sqrt(np.sum((coords - center) ** 2, axis=1))
        
        # Return coordinates within radius
        within_sphere = coords[distances <= radius]
        return set(map(tuple, np.round(within_sphere, 2)))
    
    def compute_voxel_aware_dice_coefficients(self, radius=8):
        """
        Compute Dice coefficients using voxel-size aware ROI creation
        """
        print(f"\nComputing VOXEL-AWARE Dice coefficients with {radius}mm radius spheres...")
        
        # Ensure coordinates are standardized
        self.standardize_coordinates()
        
        # Get coordinate arrays and voxel sizes
        your_coords = self.your_peaks[["x", "y", "z"]].values
        lit_coords = self.lit_peaks[["x", "y", "z"]].values
        your_voxel_sizes = self.your_peaks["voxel_size"].values
        lit_voxel_sizes = self.lit_peaks["voxel_size"].values
        
        print(f"Your data voxel sizes: {np.unique(your_voxel_sizes)}mm")
        print(f"Literature data voxel sizes: {np.unique(lit_voxel_sizes)}mm")
        
        # Create ROIs for all peaks using their respective voxel sizes
        print("Creating voxel-size aware spherical ROIs...")
        your_rois = []
        for coord, voxel_size in zip(your_coords, your_voxel_sizes):
            roi = self.create_voxel_aware_spherical_roi(coord, radius, voxel_size)
            your_rois.append(roi)
            
        lit_rois = []
        for coord, voxel_size in zip(lit_coords, lit_voxel_sizes):
            roi = self.create_voxel_aware_spherical_roi(coord, radius, voxel_size)
            lit_rois.append(roi)
        
        # Compute Dice coefficients
        print("Computing Dice coefficients...")
        dice_matrix = np.zeros((len(your_coords), len(lit_coords)))
        
        for i, your_roi in enumerate(your_rois):
            for j, lit_roi in enumerate(lit_rois):
                dice_matrix[i, j] = self.calculate_dice_coefficient(your_roi, lit_roi)
        
        # Find best matches (same as original method)
        max_dice_per_your_peak = np.max(dice_matrix, axis=1)
        best_lit_match_idx = np.argmax(dice_matrix, axis=1)
        
        # Calculate distances to best matches
        distances_to_best = []
        for i, best_j in enumerate(best_lit_match_idx):
            dist = np.sqrt(np.sum((your_coords[i] - lit_coords[best_j]) ** 2))
            distances_to_best.append(dist)
        
        results = {
            "dice_matrix": dice_matrix,
            "max_dice_per_your_peak": max_dice_per_your_peak,
            "best_lit_match_idx": best_lit_match_idx,
            "distances_to_best": np.array(distances_to_best),
            "radius": radius,
            "mean_dice": np.mean(max_dice_per_your_peak),
            "median_dice": np.median(max_dice_per_your_peak),
            "min_dice": np.min(max_dice_per_your_peak),
            "max_dice": np.max(max_dice_per_your_peak),
            "std_dice": np.std(max_dice_per_your_peak),
            "your_voxel_sizes": your_voxel_sizes,
            "lit_voxel_sizes": lit_voxel_sizes,
        }
        
        return results

In [ ]:
# Example: Compare standard vs voxel-aware analysis
def compare_analysis_methods(your_peaks_file, literature_peaks_file):
    """Compare standard 1mm resolution vs voxel-size aware analysis"""
    
    print("=== COMPARISON: Standard vs Voxel-Aware Analysis ===\n")
    
    try:
        # Standard analysis (always uses 1mm resolution)
        print("1. STANDARD ANALYSIS (1mm resolution for all data):")
        analysis_standard = HabenulaPeakAnalysis(your_peaks_file, literature_peaks_file)
        results_standard = analysis_standard.compute_all_dice_coefficients(radius=8)
        
        print(f"Standard method - Mean Dice: {results_standard['mean_dice']:.3f}")
        print(f"Standard method - Median Dice: {results_standard['median_dice']:.3f}\n")
        
        # Voxel-aware analysis (uses actual voxel sizes)
        print("2. VOXEL-AWARE ANALYSIS (respects actual voxel sizes):")
        analysis_voxel = VoxelAwareHabenulaPeakAnalysis(your_peaks_file, literature_peaks_file)
        results_voxel = analysis_voxel.compute_voxel_aware_dice_coefficients(radius=8)
        
        print(f"Voxel-aware method - Mean Dice: {results_voxel['mean_dice']:.3f}")
        print(f"Voxel-aware method - Median Dice: {results_voxel['median_dice']:.3f}\n")
        
        # Compare results
        print("3. COMPARISON:")
        dice_diff = results_voxel['mean_dice'] - results_standard['mean_dice']
        print(f"Difference in mean Dice: {dice_diff:+.3f}")
        
        if abs(dice_diff) > 0.05:
            print("⚠️  Significant difference detected! Voxel size matters for your data.")
        else:
            print("✅ Similar results - voxel size differences are minimal.")
            
        return results_standard, results_voxel
        
    except FileNotFoundError:
        print("Error: Could not find peak files.")
        print("Create example TSV files with columns: x, y, z, space, voxel_size")
        return None, None

# Uncomment to run comparison:
# results_std, results_voxel = compare_analysis_methods("your_peaks.tsv", "lit_peaks.tsv")

In [ ]:
# Helper function to create example data with different voxel sizes
def create_example_peak_data():
    """Create example TSV files with different voxel sizes for testing"""
    
    # Example your peaks (mixed voxel sizes)
    your_peaks_data = {
        'x': [2, -4, 6, -2],
        'y': [-26, -28, -24, -30], 
        'z': [-6, -4, -8, -2],
        'space': ['MNI', 'MNI', 'MNI', 'MNI'],
        'voxel_size': [1.0, 2.0, 0.7, 1.0]  # Mixed voxel sizes
    }
    
    # Example literature peaks (different voxel sizes)
    lit_peaks_data = {
        'x': [3, -3, 5, -1, 1],
        'y': [-25, -29, -23, -31, -27],
        'z': [-5, -3, -7, -1, -9],
        'space': ['MNI', 'MNI', 'MNI', 'MNI', 'MNI'],
        'voxel_size': [2.0, 1.0, 0.7, 2.0, 1.0]  # Mixed voxel sizes
    }
    
    # Save to TSV files
    your_df = pd.DataFrame(your_peaks_data)
    lit_df = pd.DataFrame(lit_peaks_data)
    
    your_df.to_csv('example_your_peaks.tsv', sep='\t', index=False)
    lit_df.to_csv('example_literature_peaks.tsv', sep='\t', index=False)
    
    print("Created example files:")
    print("- example_your_peaks.tsv (4 peaks with 0.7mm, 1mm, 2mm voxel sizes)")
    print("- example_literature_peaks.tsv (5 peaks with 0.7mm, 1mm, 2mm voxel sizes)")
    print("\nYour peaks voxel sizes:", your_peaks_data['voxel_size'])
    print("Literature peaks voxel sizes:", lit_peaks_data['voxel_size'])
    
    return 'example_your_peaks.tsv', 'example_literature_peaks.tsv'

# Uncomment to create example data and run analysis:
# your_file, lit_file = create_example_peak_data()
# results_std, results_voxel = compare_analysis_methods(your_file, lit_file)

In [ ]:
# File paths - modify these to point to your TSV files
your_peaks_file = "your_habenula_peaks.tsv"
literature_peaks_file = "literature_habenula_peaks.tsv"

try:
    # Initialize analysis
    analysis = HabenulaPeakAnalysis(your_peaks_file, literature_peaks_file)
    
    # Compute Dice coefficients with 8mm radius (standard for connectivity studies)
    results = analysis.compute_all_dice_coefficients(radius=8)
    
    # Print summary
    analysis.print_summary(results)
    
    # Create and save plots
    analysis.plot_results(results, save_path="habenula_similarity_analysis.png")
    
except FileNotFoundError as e:
    print(f"Error: Could not find file. Please check file paths.")
    print("Expected TSV format with columns: x, y, z, space (optional), voxel_size (optional)")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Optional: Test different sphere radii to see how they affect results
print("\nTesting different sphere radii:")
for radius in [6, 10, 12]:
    try:
        results_r = analysis.compute_all_dice_coefficients(radius=radius)
        print(f"Radius {radius}mm - Mean Dice: {results_r['mean_dice']:.3f}")
    except NameError:
        print("Please run the main analysis cell first to initialize the analysis object.")